In [1]:
import os 
from dotenv import load_dotenv
import logging
load_dotenv()
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger()
import re

In [2]:
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['tavily_api_key']=os.getenv('tavily_api_key')


In [3]:
from langchain_community.document_loaders import PyPDFLoader,CSVLoader,TextLoader,JSONLoader,Docx2txtLoader
from langchain_core.documents import Document


C:\Users\vinay\AppData\Local\Temp\ipykernel_15188\1361851337.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,CSVLoader,TextLoader,JSONLoader,Docx2txtLoader


In [4]:
class DocumentLoaderManager:
    def __init__(self):
        self.loader = None
        self.documents=[]

    def set_loader(self, path: str):
        """Validates the file path and initializes the appropriate LangChain loader."""
        if not os.path.exists(path):
            logging.error(f"Path does not exist: {path}")
            self.loader = None
            return None
            
        path_lower = path.lower()
        
        try:
            if path_lower.endswith('.csv'):
                self.loader = CSVLoader(path)
            elif path_lower.endswith('.pdf'):
                self.loader = PyPDFLoader(path)
            elif path_lower.endswith('.json'):
                self.loader = JSONLoader(path, jq_schema='.', text_content=False) 
            elif path_lower.endswith('.txt'):
                self.loader = TextLoader(path)
            elif path_lower.endswith('.docx'):
                self.loader=Docx2txtLoader(path)
            else:
                logging.error(f"Unsupported file format for path: {path}")
                self.loader = None
        except Exception as e:
            logging.error(f"Error initializing loader for {path}: {str(e)}")
            self.loader = None
            
        return self.loader

    def initialize_document(self,path:str):
        """Loads and returns the documents using the configured loader."""
        loader=self.set_loader(path)
        if loader is None:
            self.documents=[]
            logging.error("Loader has not been set or initialized correctly. Call set_loader() first.")
            return self.documents
        try:
            self.documents = loader.load()
            return self.documents
        except Exception as e:
            logging.exception(f"Error loading documents: {str(e)}")
            self.documents=[]
            return self.documents
    def check_toc(self,max_pages=5,max_char=3000,max_doc=50):
        toc_pattern= re.compile(
        r"^\s*(table of contents|contents|index|agenda)\s*$",
        re.IGNORECASE | re.MULTILINE,
        )
        try:
            head = "\n".join(
                doc.page_content for doc in self.documents[:max_pages]
            )[:max_char]

            has_toc = bool(self.toc_pattern.search(head))

            if has_toc and len(self.documents) < max_doc:
                return "vectorless_rag"
            return "normal_rag"
        except Exception as e :
            logging.error(f"error in selecting rag ")